Create Mel-Spectrograms from GZTAN Dataset using pipeline from preprocessing section.

In [1]:
from music_classifier.preprocessing.config import PreprocessConfig, SpectrogramConfig
from music_classifier.preprocessing.pipeline import build_spectrogram_dataset
from music_classifier.preprocessing.storage import save_dataset

In [2]:
preCon = PreprocessConfig()
speCon = SpectrogramConfig()

In [3]:
from pathlib import Path
BASE_DIR = Path.cwd().parents[3]
GTZAN_dataset_dir = BASE_DIR / "training_data" / "gtzan_dataset"
FMA_dataset_dir = BASE_DIR / "training_data" / "fma_subset"


In [10]:
records_GTZAN = list(build_spectrogram_dataset(GTZAN_dataset_dir, preCon, speCon))
records_FMA = list(build_spectrogram_dataset(FMA_dataset_dir, preCon, speCon))

[src/libmpg123/layer3.c:INT123_do_layer3():1844] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!


In [9]:
from music_classifier.preprocessing.pipeline import preprocess_file
from music_classifier.preprocessing.pipeline import build_spectrogram_record
from music_classifier.preprocessing.io import iter_audio_files

for audio_path in iter_audio_files(FMA_dataset_dir):
    try:
        audio_record = preprocess_file(audio_path, preCon)
        audio_spec = build_spectrogram_record(audio_record, speCon)
    except Exception as e:
        print(f"Failed to process {audio_path}, Error: {e}")



[src/libmpg123/layer3.c:INT123_do_layer3():1844] error: dequantization failed!
Note: Illegal Audio-MPEG-Header 0x00000000 at offset 22401.
Note: Trying to resync...
Note: Skipped 1024 bytes in input.
[src/libmpg123/parse.c:wetwork():1349] error: Giving up resync after 1024 bytes - your stream is not nice... (maybe increasing resync limit could help).
/Users/wileyjones/Desktop/CS467/CNN_Music_Classifier/src/music_classifier/preprocessing/io.py:132: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(str(path), sr=target_sr, mono=mono, dtype=dtype)
/Users/wileyjones/Desktop/CS467/CNN_Music_Classifier/.venv/lib/python3.13/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
Note: Illegal Audio-M

Failed to process /Users/wileyjones/Desktop/CS467/CNN_Music_Classifier/training_data/fma_subset/hiphop/098567.mp3, Error: segments array has zero rows — nothing to convert.
Failed to process /Users/wileyjones/Desktop/CS467/CNN_Music_Classifier/training_data/fma_subset/hiphop/098569.mp3, Error: segments array has zero rows — nothing to convert.


[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? Duh...


Failed to process /Users/wileyjones/Desktop/CS467/CNN_Music_Classifier/training_data/fma_subset/rock/108925.mp3, Error: Failed to load audio file /Users/wileyjones/Desktop/CS467/CNN_Music_Classifier/training_data/fma_subset/rock/108925.mp3: 


In [11]:
records_full = records_FMA + records_GTZAN

len(records_full)

5994

In [12]:
from music_classifier.preprocessing.splitter import stratified_split
train, val, test = stratified_split(records_full, train_ratio=.8, val_ratio=.1, test_ratio=.1)

In [13]:
print(train[0]['spectrograms'].shape)
print(len(train))

(9, 128, 130)
4794


In [15]:
train_path = Path.cwd() / "train.npz"
val_path = Path.cwd() / "val.npz"
test_path = Path.cwd() / "test.npz"

In [17]:
print(train_path)

/Users/wileyjones/Desktop/CS467/CNN_Music_Classifier/src/music_classifier/model/dataset/train.npz


In [16]:
save_dataset(train, train_path)

In [22]:
import numpy as np
train = np.load("train.npz")
print(len(train["X"]), len(train["y"]))
print(np.unique_counts(train['y']))

45642 45642
UniqueCountsResult(values=array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]), counts=array([ 800,  799, 8341, 8334, 8294,  800,  800, 8360,  800, 8314]))


In [19]:
save_dataset(val, val_path)
save_dataset(test, test_path)

In [20]:
val = np.load("val.npz")
test = np.load("test.npz")
print(len(val["X"]), len(val["y"]))
print(len(test["X"]), len(test['y']))

5673 5673
5742 5742
